In [1]:
### a second version of the main analysis, simplified and focused on our central cases

In [2]:
### Case 1 - 'perfect model'
## methods - 1.1 simple S:N, 1.2 boostrapping trends, 

## Case 2 - 'real world' (model as stand-in)
## one method - trend change-point 

### for now, do this with just one model : CESM2-WACCM

In [3]:
import s3fs
import fsspec
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys
import os
import boto3
import subprocess
import shutil
from urllib.parse import urlparse

sys.path.append('../shared/Code_examples/G6-1.5K_example/')
from Utils import weighted_annual_resample, set_time_to_center_of_bounds

import matplotlib
matplotlib.rcParams.update({'font.size': 15})

In [5]:
# --- CESM2-WACCM ---
# ICEFRAC is on the standard f09 (0.9 x 1.25 deg) atmospheric lat/lon grid — no regridding needed

# SSP2-4.5 baseline: 10-member CESM2-WACCM-SSP245 runs, publicly available on AWS (anonymous access)
# Members 001-005 extend to 2100; members 006-010 end at 2069
ssp245_base = 'ncar-cesm2-arise/CESM2-WACCM-SSP245/'
_ssp245_end2 = {m: '210012' if int(m) <= 5 else '206912' for m in [f'{i:03d}' for i in range(1, 11)]}
cesm_ssp245_ice_files = [
    [f's3://{ssp245_base}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}/atm/proc/tseries/month_1/'
     f'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}.cam.h0.ICEFRAC.201501-206412.nc',
     f's3://{ssp245_base}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}/atm/proc/tseries/month_1/'
     f'b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{m}.cam.h0.ICEFRAC.206501-{_ssp245_end2[m]}.nc']
    for m in _ssp245_end2
]

# G6-1.5K-HiLLA — r1 output is split into two time chunks; r2 and r3 are single files
cesm_HiLLA_ice_files = [
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h0.ICEFRAC.203501-203912.nc',
     's3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h0.ICEFRAC.204001-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r2/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.002.cam.h0.ICEFRAC.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r3/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.003.cam.h0.ICEFRAC.203501-208412.nc'],
]

cesm_SAI_ice_files = [
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r1/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001.cam.h0.ICEFRAC.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r2/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002.cam.h0.ICEFRAC.203501-208412.nc'],
    ['s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-SAI/r3/AMON/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.003.cam.h0.ICEFRAC.203501-208412.nc'],
]

def read_ice_from_file_list(file_list_of_lists, rename_var=None, scale=1.0, open_kwargs=None):
    """Read sea ice data for multiple members, each member given as a list of file paths.

    file_list_of_lists : list of lists — outer index = member, inner = file chunks
    rename_var         : variable name in file to rename to 'siconc'
    scale              : multiply data by this factor (use 100 to convert 0-1 fraction to %)
    open_kwargs        : extra kwargs passed to fsspec.open (e.g. {'anon': True} for public buckets)
    """
    if open_kwargs is None:
        open_kwargs = {}
    ds_list = []
    for member_files in file_list_of_lists:
        file_objects = [fsspec.open(f, mode='rb', **open_kwargs) for f in member_files]
        ds = xr.open_mfdataset([f.open() for f in file_objects],
                               combine='nested', concat_dim='time', chunks={}).load()
        # drop any duplicate time steps (e.g. overlap files)
        _, idx = np.unique(ds.time, return_index=True)
        ds = ds.isel(time=idx)
        if rename_var and rename_var in ds:
            ds = ds.rename({rename_var: 'siconc'})
        if scale != 1.0:
            ds['siconc'] = ds['siconc'] * scale
        ds_list.append(ds)
    return ds_list


def read_ice_from_directory_paths(path_list, rename_var=None, scale=1.0, time_slice=None):
    """Read sea ice data where each member lives in a directory of .nc files (e.g. UKESM)."""
    ds_list = []
    for path in path_list:
        files = fsspec.open_files(f"{path}*.nc", mode='rb')
        ds = xr.open_mfdataset([f.open() for f in files],
                               combine='nested', concat_dim='time', chunks={}).load()
        # drop any duplicate time steps (e.g. overlap files)
        _, idx = np.unique(ds.time, return_index=True)
        ds = ds.isel(time=idx)
        if rename_var and rename_var in ds:
            ds = ds.rename({rename_var: 'siconc'})
        if scale != 1.0:
            ds['siconc'] = ds['siconc'] * scale
        if time_slice:
            ds = ds.sel(time=slice(*time_slice))
        ds_list.append(ds)
    return ds_list

## CESM2-WACCM
# time_bounds gives the true start/end of each monthly interval;
# set the time coordinate to the midpoint of each interval
cesm_HiLLA_ice = read_ice_from_file_list(cesm_HiLLA_ice_files, rename_var='ICEFRAC', scale=100.0)
cesm_SAI_ice   = read_ice_from_file_list(cesm_SAI_ice_files,   rename_var='ICEFRAC', scale=100.0)

cesm_HiLLA_ice = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_HiLLA_ice]
cesm_SAI_ice   = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_SAI_ice]

## CESM2-WACCM SSP2-4.5 — CESM2-WACCM-SSP245 runs (10 members, public AWS bucket)
# anon=True for public access; files span 2015–2100 across two time chunks per member
cesm_ssp245_ice = read_ice_from_file_list(
    cesm_ssp245_ice_files,
    rename_var='ICEFRAC',
    scale=100.0,
    open_kwargs={'anon': True}
)

# correct time values to centre of bounds:
cesm_ssp245_ice   = [set_time_to_center_of_bounds(ds, time_bounds_name='time_bnds') for ds in cesm_ssp245_ice]
print(f'Loaded {len(cesm_ssp245_ice)} CESM SSP2-4.5 members, time: {cesm_ssp245_ice[0].time.values[[0, -1]]}')

Loaded 10 CESM SSP2-4.5 members, time: [cftime.DatetimeNoLeap(2015, 1, 16, 12, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2100, 12, 16, 12, 0, 0, 0, has_year_zero=True)]


In [ ]:
### DUMMY CODE, FOR PLOTTING STYLE - NEEDS UPDATING


## Fig 1 — NH sea ice area timeseries
## Rows: September, March, Annual mean   Cols: CESM2-WACCM, UKESM1-1, MIROC-ES2H, E3SMv3

season_labels = ['September', 'March', 'Annual mean']
sia_dicts     = [sep_sia_dict, mar_sia_dict, ann_sia_dict]

fig, axs = plt.subplots(3, 4, figsize=(20, 12), sharey=False)

for row, (season, sia_d) in enumerate(zip(season_labels, sia_dicts)):
    for col, model in enumerate(models):
        ax = axs[row, col]
        for scenario, da_list in sia_d[model].items():
            n     = len(da_list)
            label = f'{scenario} (n={n})' if row == 0 else None
            ts    = xr.concat(da_list, dim='member', join='inner')
            years = ts.time.values
            ax.plot(years, ts.mean('member').values, c=colours[scenario], label=label)
            ax.fill_between(years, ts.min('member').values, ts.max('member').values,
                            color=colours[f'{scenario}_light'], alpha=0.5)

        if row == 0:
            ax.set_title(model, fontsize='medium')
            ax.legend(fontsize='small', loc='upper left')
        if col == 0:
            ax.set_ylabel(f'{season}\nSIA (10⁶ km²)')
        ax.set_xlabel('Year' if row == 2 else '')
        ax.set_xlim(2020, 2084)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(ls='--', alpha=0.4)

fig.suptitle('Northern Hemisphere sea ice area', fontsize='large', y=1.01)
plt.tight_layout()
plt.savefig('Figures/seaice_area_timeseries.png', dpi=200, bbox_inches='tight')
plt.show()